# AntiDoom for Pathfinder-Eye — Colab Training

Trains a Final Token Preference Optimization (FTPO) LoRA adapter
for Ministral-3B to eliminate doom loops in conversation mode.

**Goal:** Reduce repetition loops in the "Attention" AI conversation loop.

**Hardware:** Colab T4 (free) or better. Ministral-3B fits in T4 16GB.

**Output:** A merged GGUF model you copy to the robot's SD card.

In [ ]:
# 1. Clone AntiDoom + install dependencies
!git clone https://github.com/Liquid4All/antidoom /content/antidoom
# Note: upload pathfinder-eye/antidoom/config.yaml + conversation_prompts.jsonl
# Or generate prompts from the fallback generator
!pip install antidoom/ 2>/dev/null || !pip install -e antidoom/  # TODO: test

In [ ]:
# 2. Build prompt dataset from pathfinder-eye logs
# Upload your dendrite.sqlite or leafcutter log OR use fallback
!python antidoom/build_prompts.py \
    --leafcutter-log leafcutter_conversations.log \
    --output pathfinder_convos.jsonl \
    --min-turns 3 \
    --max-prompt-len 1500

In [ ]:
# 3. Generate AntiDoom preference pairs
# (model generates completions at low temp → detector finds loops →
#  rejected token = the token that started the loop;
#  chosen tokens = filtered alternatives from the model's own logprobs)
! cd antidoom && uv run antidoom -c configs/pathfinder_ministral.yaml generate \
    --input-jsonl ../pathfinder_convos.jsonl \
    --output-jsonl ../generated_ftpo_pairs.jsonl

In [ ]:
# 4. Train LoRA adapter on the FTPO pairs
! cd antidoom && uv run antidoom -c configs/pathfinder_ministral.yaml train \
    --dataset-jsonl ../generated_ftpo_pairs.jsonl

In [ ]:
# 5. Merge adapter into the base model
! cd antidoom && uv run antidoom -c configs/pathfinder_ministral.yaml merge

# Find the merged model path
import os
merged_dirs = sorted([d for d in os.listdir('antidoom/runs') if 'HF' in d])
print(f"Merged model: antidoom/runs/{merged_dirs[-1]}")

In [ ]:
# 6. Convert to GGUF for LeafcutterLLM (use llama.cpp's conversion tools)
# This is optional — the merged model is a standard PyTorch checkpoint
# that most serving frameworks load directly. LeafcutterLLM needs GGUF.
# If your environment supports it:
!git clone https://github.com/ggerganov/llama.cpp.git /content/llama.cpp
!cd /content/llama.cpp && make -j4
!python /content/llama.cpp/convert_hf_to_gguf.py \
    antidoom/runs/pathfinder_merged \
    --outfile /content/ministral-3b-antidoom.gguf

print("Saved to ministral-3b-antidoom.gguf — copy to the robot!")

## Deploy

1. Download `ministral-3b-antidoom.gguf` to your local machine
2. `scp` it to the Pi:
   ```bash
   scp ministral-3b-antidoom.gguf pi@robot:~/the-pathfinder-eye_ai/models/
   ```
3. Point `leafcutter.service`'s `--model` at the new file (or use
   `SwapLeafcutterModel()` from the Go brain)
4. Restart: `sudo systemctl restart leafcutter`
5. Test: say "Attention" and have a long conversation — the model
   should not enter repetition loops.